# Acid–base titration curves

*chemistry · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/acid_base_titration)

Sodium hydroxide is added drop by drop to a strong acid, a weak acid and phosphoric acid. Instead of a different textbook shortcut for each region, one charge balance is solved for $[\mathrm{H^+}]$ by bisection, for every acid and every volume at once in an $(A, N)$ tensor. The curves give the equivalence points (peaks of $d\mathrm{pH}/dV$), the buffer region and the species fractions $\alpha_j(\mathrm{pH})$.

**Vector form**

$$[\mathrm{H^+}] + [\mathrm{Na^+}] = [\mathrm{OH^-}] + C_A\,\bar n,\qquad \bar n = \sum_j j\,\alpha_j,\qquad \alpha_j = \frac{\beta_j\,[\mathrm{H^+}]^{-j}}{\sum_k \beta_k\,[\mathrm{H^+}]^{-k}},\qquad \beta_j = K_1\cdots K_j$$

**Index (tensor) form**

$$f_{an} = h_{an} + \mathrm{Na}_n - \frac{K_w}{h_{an}} - C_{A,n}\,j\,\alpha_{anj} = 0 \qquad \alpha_{anj} = \frac{\beta_{aj}\,h_{an}^{-j}}{\beta_{ak}\,h_{an}^{-k}} \qquad \log_{10}\beta_{aj} = -\sum_{i\le j}\mathrm{p}K_{ai}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\mathrm{p}K_{ai}$ | `pka_table(pkas)` | (A, 3) | acid $a$, proton $i$; missing protons padded with $+\infty$ ($K = 0$) |
| $\log_{10}\beta_{aj}$ | `log_beta(pka)` | (A, 4) | cumulative constants: $-$ running sum of the pKa values, $\beta_{a0} = 1$ |
| $C_{A,n},\ \mathrm{Na}_n$ | `totals(V, c_a, V0, c_b)` | (N,) | acid and sodium after mixing $V_0$ of acid with $V_n$ of base |
| $\alpha_{anj}$ | `softmax(ln10 * (log_beta + j * pH))` | (A, N, 4) | species fractions; the sum over $k$ in the denominator is the softmax normalisation |
| $j\,\alpha_{anj}$ | `einsum('...j,j->...', alpha, j)` | (A, N) | repeated $j$ ⇒ summed: protons released per acid molecule $\bar n$ |
| $f_{an} = 0$ | `solve_ph(log_beta, C_A, Na, Kw)` | (A, N) | bisection on $\mathrm{pH} = -\log_{10}h$, all acids and volumes in one tensor |
| $\partial_V \mathrm{pH}_{an}$ | `torch.gradient(pH, spacing=(V,))` | (A, N) | slope; its peaks (found with a max-pool) are the equivalence points |

The charge balance holds everywhere along the curve, so it replaces the separate recipes for the start, the buffer region, the equivalence point and the excess of base. Every term is a monotonic function of the pH, which makes bisection safe: it always halves the bracket and cannot miss the root. A strong acid (modelled as pKa $\approx -6$) gives a sharp jump with pH 7 at equivalence, where only $\mathrm{Na^+}$, $\mathrm{Cl^-}$ and water are left. A weak acid starts higher, rises gently through the buffer region and has its equivalence point above 7, because its conjugate base $\mathrm{A^-}$ is itself a weak base. Halfway to equivalence $[\mathrm{HA}] = [\mathrm{A^-}]$, so Henderson–Hasselbalch, $\mathrm{pH} = \mathrm{p}K_a + \log_{10}([\mathrm{A^-}]/[\mathrm{HA}])$, gives $\mathrm{pH} = \mathrm{p}K_a$: a titration is how pKa values are measured. Phosphoric acid (pKa ≈ 2.15, 7.20 and 12.35, approximate values at 25 °C) shows two jumps; the third proton is so weakly acidic that 0.1 M NaOH cannot pull it off visibly, and its jump is lost in the excess of base. The species panel shows why: each $\alpha_j$ peaks between two pKa values, and two neighbours are equal exactly at $\mathrm{pH} = \mathrm{p}K$. Simplifications: concentrations instead of activities, 25 °C, no CO₂ from the air.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/acid_base_titration/main.py
"""Main loop: acids and volumes → charge balance solved for pH → equivalence points, buffer check → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    c_a, c_b, v0, pka_weak = params["c_acid"], params["c_base"], params["V0"], params["pKa_weak"]
    names, pkas = tensors.acids(pka_weak)
    log_beta = tensors.log_beta(tensors.pka_table(pkas, dev))  # (A, 4)
    v_eq = c_a * v0 / c_b  # mL of base per proton

    V = tensors.volumes(params["V_max"], 201 if fast_mode() else 1601, dev)  # (N,)
    C_A, Na = tensors.totals(V, c_a, v0, c_b)  # (N,), (N,)
    pH = solve.solve_ph(log_beta, C_A, Na, tensors.KW)  # (A, N): every acid and volume at once
    dpdv = solve.slope(pH, V)  # (A, N)
    found = solve.equivalence_points(dpdv, V)
    residual = equation.charge_balance(pH, log_beta, C_A, Na, tensors.KW).abs().max().item()
    print(f"{len(names)} acids × {len(V)} volumes solved together; largest charge-balance residual {residual:.1e} M")
    print(f"expected equivalence volumes: {v_eq:g} mL per proton (c_a V₀ / c_b)")

    # exact pH at a few key volumes: half-equivalence and equivalence points
    V_key = torch.tensor([0.5, 1.0, 1.5, 2.0], dtype=torch.float64, device=dev) * v_eq
    pH_key = solve.solve_ph(log_beta, *tensors.totals(V_key, c_a, v0, c_b), tensors.KW)  # (A, 4)
    for a, name in enumerate(names):
        peaks = ", ".join(f"{v:.2f}" for v in found[a]) or "none"
        print(f"{name:<20} start pH {pH[a, 0].item():5.2f}   dpH/dV peaks at {peaks} mL")
        print(
            f"{'':<20} pH at ½·V_eq = {pH_key[a, 0].item():.3f}   at V_eq = {pH_key[a, 1].item():.3f}   "
            f"at 2·V_eq = {pH_key[a, 3].item():.3f}"
        )

    # buffer region of the weak acid: Henderson–Hasselbalch vs the exact curve, from 10 % to 90 % of V_eq
    buffer = (V >= 0.1 * v_eq) & (V <= 0.9 * v_eq)
    V_buf = V[buffer]
    pH_hh = equation.henderson_hasselbalch(pka_weak, c_b * V_buf, c_a * v0 - c_b * V_buf)
    print(f"buffer region of the weak acid: |pH − HH| ≤ {(pH[1, buffer] - pH_hh).abs().max().item():.3f}")

    # species fractions of H₃PO₄ over the whole pH scale
    pH_grid = torch.linspace(0.0, 14.0, 141 if fast_mode() else 561, dtype=torch.float64, device=dev)
    alpha = equation.fractions(pH_grid[None], log_beta[2:3])[0]  # (P, 4)
    print(f"α fractions of H₃PO₄ sum to 1 within {(alpha.sum(-1) - 1).abs().max().item():.1e}")

    return plot.figure(V, pH, dpdv, found, names, v_eq, pka_weak, V_buf, pH_hh, pH_grid, alpha, pkas[2], out_dir)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/acid_base_titration/equation.py
r"""Chemistry → tensor translation.

An acid H_nA gives up its protons one at a time with constants K₁, K₂, …. With h = [H⁺] the fraction of the acid
that has lost j protons is

    α_{aj}(h) = β_{aj} h^{-j} / Σ_k β_{ak} h^{-k},        β_{aj} = K_{a1} K_{a2} ⋯ K_{aj}   (β_{a0} = 1)

and the average charge carried per acid molecule is  n̄_a = j α_{aj}   (summed over j).

The solution must be electrically neutral (charge balance): positive charges = negative charges,

    h + [Na⁺] = K_w / h + C_A n̄(h)       ⇔      f(h) = h + [Na⁺] - K_w/h - C_A j α_j(h) = 0

for every acid a and titrant volume n. Repeated j is summed. Everything is done in log₁₀ h = -pH, because
h spans 15 orders of magnitude; the α are a softmax over j of log β_j + j pH.
"""

import math

import torch

LN10 = math.log(10.0)


def fractions(pH: torch.Tensor, log_beta: torch.Tensor) -> torch.Tensor:
    """α_{a…j} = β_{aj} h^{-j} / Σ_k β_{ak} h^{-k}.   pH: (A, …), log β: (A, J) → (A, …, J).

    log₁₀(β_j h^{-j}) = log₁₀ β_j + j pH, and dividing by the sum is a softmax (stable even for h^{-3} ~ 10⁴²).
    """
    j = torch.arange(log_beta.shape[-1], dtype=pH.dtype, device=pH.device)  # (J,) protons lost
    shape = (log_beta.shape[0],) + (1,) * (pH.dim() - 1) + (log_beta.shape[-1],)
    logits = LN10 * (log_beta.reshape(shape) + j * pH[..., None])  # (A, …, J)
    return torch.softmax(logits, dim=-1)


def mean_charge(alpha: torch.Tensor) -> torch.Tensor:
    """n̄ = j α_j: protons released per acid molecule.   Repeated j ⇒ summed.   (…, J) → (…)."""
    j = torch.arange(alpha.shape[-1], dtype=alpha.dtype, device=alpha.device)
    return torch.einsum("...j,j->...", alpha, j)


def charge_balance(pH: torch.Tensor, log_beta: torch.Tensor, c_acid: torch.Tensor, sodium: torch.Tensor, kw: float):
    """f = h + [Na⁺] - K_w/h - C_A n̄.   pH: (A, N); C_A, [Na⁺]: (N,) → (A, N).

    f > 0: too many positive charges, so the true pH is higher; f decreases monotonically as pH rises.
    """
    h = 10.0 ** (-pH)
    return h + sodium - kw / h - c_acid * mean_charge(fractions(pH, log_beta))


def henderson_hasselbalch(pka: float, base: torch.Tensor, acid: torch.Tensor) -> torch.Tensor:
    """Buffer estimate pH ≈ pKa + log₁₀([A⁻]/[HA]), with [A⁻]/[HA] ≈ (added base)/(acid left)."""
    return pka + torch.log10(base / acid)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/acid_base_titration/tensors.py
"""Build the concrete tensors: the acids (their pKa values), titrant volumes and total concentrations.

Units: concentrations in mol/L (M), volumes in mL. Every acid is padded to three protons; a missing proton gets
pKa = +∞ (Kₐ = 0), so it never comes off and the same formulas cover mono- and triprotic acids.
"""

import math

import torch

KW = 1e-14  # ion product of water at 25 °C


def acids(weak_pKa: float) -> tuple[list[str], list[list[float]]]:
    """Names and pKa lists. pKa values of H₃PO₄ are approximate textbook values at 25 °C."""
    return (
        ["HCl (strong)", f"weak acid, pKa {weak_pKa:g}", "H₃PO₄ (triprotic)"],
        [[-6.0], [weak_pKa], [2.15, 7.20, 12.35]],  # HCl: pKa ≈ -6, effectively fully dissociated
    )


def pka_table(pkas: list[list[float]], device: torch.device) -> torch.Tensor:
    """pKa_{aj} of shape (A, 3), padded with +∞ for protons an acid does not have."""
    rows = [p + [math.inf] * (3 - len(p)) for p in pkas]
    return torch.tensor(rows, dtype=torch.float64, device=device)


def log_beta(pka: torch.Tensor) -> torch.Tensor:
    """log₁₀ β_{aj} = log₁₀(K_{a1} ⋯ K_{aj}) = -(pK_{a1} + … + pK_{aj}),  β_{a0} = 1.   (A, 3) → (A, 4).

    A missing proton (pKa = +∞) gives log β = -∞, i.e. that species never forms.
    """
    zero = torch.zeros_like(pka[:, :1])
    return torch.cat([zero, -pka.cumsum(-1)], dim=-1)


def volumes(v_max: float, n: int, device: torch.device) -> torch.Tensor:
    """Titrant volumes V_n in mL, shape (N,)."""
    return torch.linspace(0.0, v_max, n, dtype=torch.float64, device=device)


def totals(V: torch.Tensor, c_acid: float, v0: float, c_base: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Total acid C_A(V) = c_a V₀/(V₀+V) and sodium [Na⁺](V) = c_b V/(V₀+V) after mixing, both (N,) in M."""
    total_volume = v0 + V
    return c_acid * v0 / total_volume, c_base * V / total_volume


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/acid_base_titration/solve.py
"""Solve the charge balance for the pH by bisection, then find the equivalence points from dpH/dV."""

import torch
import torch.nn.functional as F

from .equation import charge_balance


def solve_ph(
    log_beta: torch.Tensor,
    c_acid: torch.Tensor,
    sodium: torch.Tensor,
    kw: float,
    lo: float = -2.0,
    hi: float = 16.0,
    iterations: int = 60,
) -> torch.Tensor:
    """pH_{an} for every acid a and volume n, shape (A, N), by bisection on pH = -log₁₀ h.

    f(pH) decreases with pH, so f(mid) > 0 means the root lies above mid. Each step halves the bracket for ALL
    (a, n) pairs at once; 60 steps shrink 18 pH units to ~10⁻¹⁷.
    """
    shape = (log_beta.shape[0], c_acid.shape[0])
    low = torch.full(shape, lo, dtype=c_acid.dtype, device=c_acid.device)
    high = torch.full(shape, hi, dtype=c_acid.dtype, device=c_acid.device)
    for _ in range(iterations):
        mid = 0.5 * (low + high)
        above = charge_balance(mid, log_beta, c_acid, sodium, kw) > 0
        low = torch.where(above, mid, low)
        high = torch.where(above, high, mid)
    return 0.5 * (low + high)


def slope(pH: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """dpH/dV by central differences along the volume axis.   (A, N) → (A, N) in pH units per mL."""
    return torch.gradient(pH, spacing=(V,), dim=-1)[0]


def equivalence_points(dpdv: torch.Tensor, V: torch.Tensor, window_ml: float = 2.0, threshold: float = 0.3):
    """Volumes where dpH/dV has a peak: a local maximum over ±window_ml that is steeper than threshold pH/mL.

    A max-pool finds every local maximum in one call; the first and last window are skipped (edges, not peaks).
    Returns one list of volumes per acid.
    """
    k = max(1, int(round(window_ml / (V[1] - V[0]).item())))
    pooled = F.max_pool1d(dpdv[:, None], 2 * k + 1, stride=1, padding=k)[:, 0]  # (A, N) local maxima
    peak = (dpdv == pooled) & (dpdv > threshold)
    peak[:, :k] = False
    peak[:, -k:] = False
    return [V[row].tolist() for row in peak]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/acid_base_titration/plot.py
"""Static figure: titration curves with equivalence points, the slope dpH/dV, and the H₃PO₄ species fractions."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import save_png

SPECIES = ["H₃PO₄", "H₂PO₄⁻", "HPO₄²⁻", "PO₄³⁻"]


def figure(
    V: torch.Tensor,
    pH: torch.Tensor,
    dpdv: torch.Tensor,
    found: list,
    names: list[str],
    v_eq: float,
    pka_weak: float,
    V_buf: torch.Tensor,
    pH_hh: torch.Tensor,
    pH_grid: torch.Tensor,
    alpha: torch.Tensor,
    pka_poly: list[float],
    out_dir: Path,
) -> Path:
    v, p, d = V.cpu().numpy(), pH.cpu().numpy(), dpdv.cpu().numpy()
    fig, (ax_p, ax_d, ax_a) = plt.subplots(1, 3, figsize=(12.5, 4.3), gridspec_kw={"width_ratios": [1.2, 1, 1]})

    # --- (a) pH curves ---
    for a, name in enumerate(names):
        ax_p.plot(v, p[a], lw=2, color=f"C{a}", label=name)
        for ve in found[a]:
            ax_p.plot(ve, np.interp(ve, v, p[a]), "o", ms=7, mfc="white", mew=2, color=f"C{a}")
    ax_p.plot(V_buf.cpu().numpy(), pH_hh.cpu().numpy(), "--", lw=1.5, color="#111827", label="Henderson–Hasselbalch")
    p_half = np.interp(v_eq / 2, v, p[1])  # computed weak-acid pH halfway to equivalence
    ax_p.plot(v_eq / 2, p_half, "s", ms=6, color="#f59e0b", label=f"½·V_eq: pH {p_half:.2f} ≈ pKa {pka_weak:g}")
    ax_p.axhline(7, color="#9ca3af", ls=":", lw=1)
    ax_p.set_xlim(v[0], v[-1])
    ax_p.set_ylim(0, 14)
    ax_p.set_xlabel("NaOH added V (mL)")
    ax_p.set_ylabel("pH")
    ax_p.set_title("titration curves (○ = equivalence point found)", fontsize=10)
    ax_p.legend(fontsize=7.5, loc="lower right")

    # --- (b) slope: peaks mark the equivalence points ---
    for a in range(len(names)):
        ax_d.plot(v[1:-1], np.maximum(d[a, 1:-1], 1e-3), lw=1.5, color=f"C{a}")
    for k in range(1, 4):
        if k * v_eq <= v[-1]:
            ax_d.axvline(k * v_eq, color="#9ca3af", ls="--", lw=1)
            ax_d.text(k * v_eq, 1.5e-3, f"{k}·V_eq ", ha="right", fontsize=8, color="#6b7280")
    ax_d.set_yscale("log")
    ax_d.set_xlim(v[0], v[-1])
    ax_d.set_ylim(1e-3, None)
    ax_d.set_xlabel("V (mL)")
    ax_d.set_ylabel("dpH/dV (per mL)")
    ax_d.set_title("slope of the curve", fontsize=10)

    # --- (c) species fractions of H₃PO₄ ---
    g, al = pH_grid.cpu().numpy(), alpha.cpu().numpy()
    for j, label in enumerate(SPECIES):
        ax_a.plot(g, al[:, j], lw=2, color=f"C{j + 3}", label=label)
    for pk in pka_poly:
        ax_a.axvline(pk, color="#9ca3af", ls="--", lw=1)
        ax_a.text(pk, 1.03, f"{pk:g}", ha="center", fontsize=8, color="#6b7280")
    ax_a.set_xlim(0, 14)
    ax_a.set_ylim(0, 1.1)
    ax_a.set_xlabel("pH")
    ax_a.set_ylabel(r"fraction $\alpha_j$")
    ax_a.set_title("H₃PO₄ species (pKa ≈ 2.15, 7.20, 12.35)", fontsize=10)
    ax_a.legend(fontsize=8, loc="center right")

    fig.tight_layout()
    return save_png(fig, out_dir / "acid_base_titration.png")


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'c_acid': 0.1,
    'V0': 25.0,
    'c_base': 0.1,
    'V_max': 80.0,
    'pKa_weak': 4.76
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.acid_base_titration")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.acid_base_titration.main")
problem_dir = Path("problems/chemistry/acid_base_titration")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
